# LLM Transcript Cleaner

This notebook demonstrates how to clean transcript segment text using the local vLLM server (running `Qwen2.5-VL-7B-Instruct-NVFP4`). It reads JSON files from `data/transcribe_result/`, sends each segment's text to the LLM to fix grammar/filler words, and saves the cleaned text back into the JSON.

In [ ]:
import os
import json
import csv
from pathlib import Path
from openai import OpenAI
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# ---- Configuration ----
TRANSCRIPTS_DIR = Path("../data/transcribe_result")
CSV_PATH = Path("../data/cleaned_files.csv")
LLM_BASE_URL = "http://localhost:8001/v1"
LLM_MODEL = "Qwen2.5-VL-7B-Instruct-NVFP4"
# # Only for current dataset which is @Reformed21 youtube channel
# SYSTEM_PROMPT = """
# You are an expert transcript cleaner for Indonesian, English, and Mandarin sermons/speeches.
# 
# Your task is to clean and correct raw ASR (speech-to-text) transcripts.
# 
# Instructions:
# 1. Fix spelling, punctuation, capitalization, and grammar.
# 2. Remove filler words, false starts, and unnecessary repetitions.
# 3. Correct obvious transcription errors caused by speech recognition (e.g., misheard words, names, or phrases), especially:
#    - Biblical names, places, and theological terms
#    - Common sermon vocabulary
# 4. Use context to infer the correct word when the original is clearly wrong or nonsensical.
# 5. DO NOT change meaning, add new information, or paraphrase unnecessarily.
# 6. If a word or phrase is ambiguous and cannot be confidently corrected, keep the original.
# 7. Preserve the original language mix (Indonesian, English, Mandarin) unless clearly incorrect.
# 8. Keep formatting simple and readable (natural sentence structure).
# 
# Output rules:
# - Respond ONLY with the cleaned and corrected text.
# - Do NOT include explanations, notes, or annotations.
# - If the text is already clean, return it unchanged.
# """

# For a more general-purpose cleaner
SYSTEM_PROMPT = """
You are an expert transcript cleaner for Indonesian, English, and Mandarin speeches.

Your task is to clean and correct the provided audio transcript by:
- Fixing spelling, punctuation, capitalization, and grammar
- Removing filler words, disfluencies, and unnecessary repetitions
- Correcting obvious transcription errors from speech-to-text systems (e.g., misheard words)

IMPORTANT RULES:
- Preserve the original meaning, tone, and intent at all times
- You MAY correct incorrect words or names IF the correct form is clear from context (e.g., common biblical names, places, or well-known terms)
- Do NOT guess or hallucinate corrections if uncertain — keep the original word in that case
- Do NOT add new information, explanations, or rephrase beyond necessary correction
- Keep the structure natural and readable as spoken text

If the text is already clean, return it unchanged.

Respond ONLY with the cleaned text, without any additional commentary.
"""
MAX_CONCURRENT_REQUESTS = 32

# ---- Initialize LLM Client ----
llm_client = OpenAI(base_url=LLM_BASE_URL, api_key="not-needed")

def load_cleaned_ids():
    if not CSV_PATH.exists():
        return set()
    with open(CSV_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader, None) # skip header
        return {row[0] for row in reader if row}

def add_cleaned_id(file_id):
    file_exists = CSV_PATH.exists()
    with open(CSV_PATH, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["file_id"])
        writer.writerow([file_id])

def clean_transcript_text(text: str) -> str:
    if not text.strip():
        return text
    
    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT.strip()},
            {"role": "user", "content": text}
        ],
        temperature=0.1, # Low temperature for consistent cleanup
        max_tokens=500
    )
    return response.choices[0].message.content.strip()

/home/crc/Documents/david/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def process_single_segment(segment):
    """Worker function for threading: cleans a single segment."""
    original_text = segment.get("text", "")
    if original_text:
        try:
            return segment, clean_transcript_text(original_text)
        except Exception as e:
            tqdm.write(f"Error processing segment: {e}")
            return segment, None
    return segment, None

def process_transcript_file(file_path: Path):
    file_id = file_path.stem
    
    # Check CSV to see if already processed
    cleaned_ids = load_cleaned_ids()
    if file_id in cleaned_ids:
        tqdm.write(f"Skipping {file_path.name}: already marked as cleaned in CSV.")
        return file_path

    # Read original data
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    segments = data.get("segments", [])
    
    tqdm.write(f"Processing {file_path.name} ({len(segments)} segments)")
    
    # Use ThreadPoolExecutor to call LLM API concurrently
    with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_REQUESTS) as executor:
        # Submit all tasks
        futures = {executor.submit(process_single_segment, seg): seg for seg in segments}
        
        # Process results as they complete
        for future in tqdm(as_completed(futures), total=len(segments), desc="Segments", leave=False):
            segment, cleaned_text = future.result()
            if cleaned_text is not None:
                # Overwrite 'text' directly instead of adding a new key
                segment["text"] = cleaned_text
                
    # Write modified data back to the same file
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
            
    tqdm.write(f"Finished processing. Saved to: {file_path.name}")
    
    # Log to CSV
    add_cleaned_id(file_id)
    tqdm.write(f"Registered {file_id} to CSV.\n")
    return file_path

In [ ]:
# Example: process only the first JSON file that hasn't already been processed
cleaned_ids = load_cleaned_ids()
json_files = [f for f in TRANSCRIPTS_DIR.glob("*.json") if f.stem not in cleaned_ids and not f.name.endswith(".backup")]

if json_files:
    test_file = json_files[0]
    out_file = process_transcript_file(test_file)
    
    # Print results from the file
    if out_file and out_file.exists():
        with open(out_file, "r", encoding="utf-8") as f:
            cleaned_data = json.load(f)
            print("\nPreview of cleaned segments:")
            for i, seg in enumerate(cleaned_data.get("segments", [])[:5]):
                print(f"  [{seg.get('start')} - {seg.get('end')}] {seg.get('text')}")
else:
    print("No eligible transcript JSON files found.")

Cleaned file already exists: l3PH0jN8DoU_cleaned.json. Skipping.

Preview of cleaned segments:
  [0 - 30] Intro: Sudara-sudara kita akan membaca Kitab Suci dari 1 Yohannes Fasal ke-2. Kita membaca dari ayat yang ke-3 sampai dengan ayat yang ke-6.
  [30 - 60] Satu yang ada di fasal ke-2, dari ayat ke-3 sampai dengan ayat ke-6. Mari kita membaca bersama-sama ayat ke-3. Dan inilah tandanya, bahwa kita mengenal Allah. Yaitu jika kita menuruti perintah-perintahnya. Barang siapa berkata, "Aku mengenal dia," tapi ia tidak menuruti perintahnya. Ia adalah seorang pendusta dan di dalamnya tidak ada kebenaran. Tetapi
  [60 - 90] Barang siapa menuruti firman-Nya, di dalam orang itu sungguh sudah sempurna kasih Allah. Dengan itulah kita ketahui bahwa kita ada di dalam Dia. Barang siapa mengatakan bahwa ia ada di dalam Dia, barangsiapa mengatakan bahwa ia ada di dalam Dia, ia wajib hidup sama seperti Krisus telah hidup atau terjemahan lain barangsiapa menuruti firman-Nya di dalam orang itu.
  [90 - 

In [4]:
# Clean all files in the directory
if json_files:
    print(f"Total files to process: {len(json_files)}")
    for json_file in tqdm(json_files, desc="Overall Progress", unit="file"):
        process_transcript_file(json_file)
else:
    print("No files to process.")

print("All done!")

Overall Progress: 100%|██████████| 2456/2456 [3:45:30<00:00,  5.51s/file]  

All done!
